# Lean-13e : Le Théorème du Libre Arbitre (Conway-Kochen)

**Navigation** : [<< Lean-16b FRACTRAN](Lean-16b-Conway-FRACTRAN-Lean-Native.ipynb) | [Index](README.md) | [Lean-13f Correlation Bell du vide >>](Lean-13f-Correlation-Bell-Vide-Lean-Python.ipynb)

**Kernel** : Python 3 (illustrations conceptuelles) + Lean 4 via WSL (exécution du port formel)

***

### Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Expliquer le pont entre le théorème de Kochen-Specker et l'exclusion du determinisme a variables cachees.
2. Définir les trois axiomes SPIN, TWIN et MIN et leur rôle dans l'argument en deux temps.
3. Distinguer ce que le théorème dit de ce qu'il ne dit pas (avertissement pedagogique).
4. Lire et interpreter la chaîne de reduction formelle `free_will_theorem` -> `kochen_specker` en Lean 4.

## Introduction

En 2006, John Horton Conway et Simon Kochen publient *The Free Will Theorem*, l'un des
résultats les plus surprenants -- et les plus mal compris -- de la physique mathematique
moderne. Son enonce, volontairement provocateur :

> Si les experimentateurs disposent d'un libre arbitre (leurs choix de mesure ne sont pas une
> fonction du passe), alors les particules élémentaires en disposent aussi : leur reponse a une
> mesure n'est pas une fonction de tout ce qui s'est passe avant.

Ce notebook est le **troisieme volet** de l'hommage a Conway dans cette serie (après
[Lean-16](Lean-16-Conway-Homme-et-Oeuvre-Lean-Python.ipynb), l'homme et l'oeuvre, et
[GOL-01](GOL/GOL-01-Regles-Vie-Lean.ipynb), le Game of Life). La ou
[Lean-13](Lean-13-Kochen-Specker.ipynb) etablit le **noyau combinatoire** (le théorème de
Kochen-Specker, table Cabello a 18 vecteurs), ce notebook se concentre **en profondeur** sur le
théorème de libre arbitre lui-même :

1. Le **pont** depuis Kochen-Specker : pourquoi l'absence de coloration valide borne le determinisme.
2. Les **trois axiomes** SPIN / TWIN / MIN (FIN), enonces physiques et formalisation.
3. L'**argument en deux temps** (une particule, puis deux particules entrelacees).
4. Le **cadre conceptuel** : ce que le théorème dit, et surtout ce qu'il **ne dit pas**.
5. L'**adossement au port Lean 4** : `FreeWillTheorem.lean` et `KochenSpecker.lean`, executes pour de vrai.
6. Une structure d'**extensibilite** vers d'autres résultats de Conway.

### Prerequis
- [Lean-13 Kochen-Specker](Lean-13-Kochen-Specker.ipynb) (le noyau combinatoire -- fortement recommande)
- Notions d'algebre linéaire (bases orthogonales) et de mecanique quantique élémentaire (spin, intrication)
- Notebooks Lean-1 a Lean-7 pour les aspects formels

### Duree estimée : 60 minutes

## Plan

1. [Du théorème de Kochen-Specker au libre arbitre](#1)
2. [Les trois axiomes : SPIN, TWIN, MIN](#2)
3. [L'enonce du théorème -- ce qu'il dit et ne dit pas](#3)
4. [Le port Lean 4 : adossement au code réel](#4)
5. [La chaîne de reduction formelle](#5)
6. [La non-localite quantitative : la frontiere CHSH](#6)
7. [Extensibilite : vers d'autres résultats de Conway](#7)
8. [Exercices](#8)


<a id="1"></a>
## 1. Du Théorème de Kochen-Specker au Libre Arbitre

### 1.1 Rappel : ce que Kochen-Specker interdit

Le notebook [Lean-13](Lean-13-Kochen-Specker.ipynb) etablit le **théorème de Kochen-Specker**
(version Cabello a 18 vecteurs) : il n'existe **aucune** fonction `c : vecteurs -> {0,1}` qui
attribue exactement un `1` par base orthogonale. Formellement, dans `KochenSpecker.lean` :

```lean
theorem kochen_specker : ¬ ∃ c : Coloring, IsValidColoring c
```

Autrement dit : on ne peut pas pre-assigner de maniere coherente une valeur 0/1 a chaque
observable de spin, independamment du contexte de mesure. C'est l'exclusion des **variables
cachees non-contextuelles**.

### 1.2 Le saut conceptuel de Conway-Kochen

Conway et Kochen transforment ce résultat combinatoire en un enonce sur le **determinisme**.
L'idee : un univers déterministe (a variables cachees) devrait fixer, une fois l'état cache `lambda`
connu, la reponse de chaque particule a chaque mesure possible. Cette reponse serait une
**fonction** `f(lambda, direction) -> {0,1}`.

Mais si l'on impose les règles physiques de la mecanique quantique (l'axiome SPIN ci-dessous),
cette fonction devrait etre une coloration valide au sens de Kochen-Specker. Or il n'en existe
aucune. **Donc la reponse de la particule ne peut pas etre une fonction du passe** : en ce sens
mathematique précis, elle est "libre".

Le reste du notebook deroule cet argument rigoureusement, puis l'adosse au port Lean 4.


In [1]:
import subprocess
from pathlib import Path
from lean_notebook_utils import (
    find_lean_project, get_lean_project_path,
    run_lake, count_sorry, is_native_platform, is_windows
)

# Cross-platform: native path on Linux/macOS, WSL path on Windows
LEAN_PROJECT = get_lean_project_path("GOL/conway_lean")
WIN_LEAN_PROJECT = find_lean_project("GOL/conway_lean")
WSL_LEAN_PROJECT = LEAN_PROJECT  # Already WSL-formatted on Windows

assert (WIN_LEAN_PROJECT / "lakefile.lean").exists(), "conway_lean/lakefile.lean introuvable"
assert (WIN_LEAN_PROJECT / "Conway" / "FreeWillTheorem.lean").exists(), "FreeWillTheorem.lean introuvable"
assert (WIN_LEAN_PROJECT / "Conway" / "KochenSpecker.lean").exists(), "KochenSpecker.lean introuvable"

print("Projet Lean detecte :")
print("  chemin            : .../conway_lean")
print("  modules cibles    : Conway.KochenSpecker, Conway.FreeWillTheorem")
plat = "native" if is_native_platform() else "WSL"
print(f"  plateforme        : {plat}")
print("Setup OK : projet conway_lean accessible")


Projet Lean detecte :
  chemin            : .../conway_lean
  modules cibles    : Conway.KochenSpecker, Conway.FreeWillTheorem
  plateforme        : WSL
Setup OK : projet conway_lean accessible


<a id="2"></a>
## 2. Les Trois Axiomes : SPIN, TWIN, MIN

Le théorème de libre arbitre ne sort pas du neant : il decoule de **trois axiomes physiques**,
tous très modestes et bien etablis experimentalement. Les voici, avec leur formalisation dans
`FreeWillTheorem.lean`.


### 2.1 SPIN -- la règle du "101"

**Enonce physique.** Pour une particule de spin 1, si l'on mesure le **carré** de la composante
de spin selon trois directions mutuellement orthogonales, on obtient toujours les valeurs
{1, 0, 1} dans un certain ordre (exactement un `0`). Dans le cadre a 4 dimensions de la table
Cabello (paires entrelacees), cela devient : exactement un `1` par base orthogonale de 4 vecteurs.

**C'est exactement la contrainte de Kochen-Specker.** Une reponse déterministe `f(lambda, .)` qui
respecte SPIN est, pour chaque état cache `lambda`, une coloration valide au sens `IsValidColoring`.

Formalisation Lean (`FreeWillTheorem.lean`) :

```lean
abbrev DeterministicResponse := HiddenState -> VecIdx -> Bool

def SatisfiesSPIN (f : DeterministicResponse) : Prop :=
  ∀ state : HiddenState, IsValidColoring (f state)
```

Le pont avec Kochen-Specker est donc **direct** : SPIN = "chaque état cache induit une coloration valide".


In [2]:
# Illustration : une reponse deterministe respectant SPIN = une coloration KS valide.
# Les 9 contextes (bases orthogonales) de la table Cabello a 18 vecteurs (cf Lean-13, section 3) :
contexts = [
    (0, 1, 2, 3), (0, 4, 5, 6), (7, 8, 2, 9), (7, 10, 6, 11), (1, 4, 12, 13),
    (8, 10, 13, 14), (15, 16, 3, 9), (15, 17, 5, 11), (16, 17, 12, 14),
]

def is_valid_coloring(response):
    """SatisfiesSPIN pour un état cache : exactement un '1' par contexte (cf IsValidColoring)."""
    return all(sum(response[i] for i in ctx) == 1 for ctx in contexts)

# Echantillonnage de reponses deterministes "candidates" (une par état cache lambda).
import random
random.seed(0)
trials = 200_000
n_valid = sum(1 for _ in range(trials)
              if is_valid_coloring([random.randint(0, 1) for _ in range(18)]))

print(f"Reponses deterministes echantillonnees : {trials:,}")
print(f"Reponses respectant SPIN (coloration valide)  : {n_valid}")
print()
print("Aucune reponse deterministe ne peut respecter SPIN sur les 18 vecteurs :")
print("c'est precisement le contenu de `fwt_single_particle` (reduction a Kochen-Specker).")
print("La preuve formelle exhaustive (2^18) est dans Lean-13 ; ici on illustre le cadrage SPIN.")


Reponses deterministes echantillonnees : 200,000
Reponses respectant SPIN (coloration valide)  : 0

Aucune reponse deterministe ne peut respecter SPIN sur les 18 vecteurs :
c'est precisement le contenu de `fwt_single_particle` (reduction a Kochen-Specker).
La preuve formelle exhaustive (2^18) est dans Lean-13 ; ici on illustre le cadrage SPIN.


### Interpretation : 0 coloration valide sur 200 000 echantillons

Le résultat **0 valide / 200 000** confirme experimentalement le théorème KS. Parmi les $2^{18} = 262\,144$ colorations candidates possibles, **aucune** ne satisfait la contrainte : chaque vecteur apparait dans exactement un contexte colore. L'impossibilite n'est pas une question de probabilite -- elle est **structurelle**.

| Paramètre | Valeur | Signification |
|-----------|--------|---------------|
| Echantillons | 200 000 | Tirages **avec remise** : ~53 % de réponses distinctes |
| Colorations valides | 0 | Aucune ne satisfait SPIN |
| Espace total | 262 144 | $2^{18}$ reponses déterministes |

L'echantillonnage tire chaque reponse **avec remise** : sur 200 000 tirages, le nombre de
reponses *distinctes* attendu est $2^{18}\,(1 - e^{-200000/2^{18}}) \approx 139\,900$, soit
~53 % de l'espace -- et non ~76 % ($200000/2^{18}$), qui compterait chaque tirage repete
comme une couverture nouvelle.

> **Note technique** : la preuve formelle exhaustive (2^18) est dans Lean-13. L'echantillonnage ici est une illustration pedagogique du cadrage SPIN.

### 2.2 TWIN -- la correlation des jumelles

**Enonce physique.** On prepare deux particules de spin 1 dans un état **intrique** (singulet),
envoyees a deux experimentateurs spatialement separes, Alice et Bob. Si les deux mesurent le
carré du spin selon la **même** direction, ils obtiennent **le même résultat**. C'est la
correlation EPR, vérifiée experimentalement.

Consequence cruciale : dans un modèle déterministe, Alice et Bob doivent partager **la même
fonction de reponse**. Ce que l'un repondrait selon la direction `d`, l'autre le repond aussi.

Formalisation Lean :

```lean
abbrev TwoParticleResponse := HiddenState -> Experimenter -> VecIdx -> Bool

def SatisfiesTWIN (f : TwoParticleResponse) : Prop :=
  ∀ state : HiddenState, ∀ dir : VecIdx,
    f state .alice dir = f state .bob dir
```


### 2.3 MIN (FIN) -- l'indépendance des choix

**Enonce physique.** Les choix de direction de mesure d'Alice et de Bob sont **independants** :
comme ils sont spatialement separes (separation de genre espace), aucun signal ne peut relier le
choix de l'un a la reponse de l'autre. La reponse d'Alice ne depend que de **sa propre** direction,
pas de celle de Bob.

- **Version 2006 (FIN)** : l'information ne se propage pas plus vite que la lumiere (causalite relativiste).
- **Version 2009 (MIN)** : reformulation plus propre -- la reponse de chaque experimentateur ne
  depend pas du choix de l'autre. C'est cette version que l'on formalise.

**Formalisation : MIN est structurel.** Dans `FreeWillTheorem.lean`, MIN n'est pas un predicat
separe : il est encode dans la **signature de type**. La fonction `f state e dir` prend la
direction de l'experimentateur `e`, mais **jamais** celle de l'autre. L'indépendance est donc
garantie par construction.

```lean
structure SatisfiesFWT (f : TwoParticleResponse) : Prop where
  spin : ∀ e : Experimenter, SatisfiesSPIN (f . e)
  twin : SatisfiesTWIN f
  -- MIN : structurel (la signature de f n'expose pas la direction de l'autre experimentateur)
```


<a id="3"></a>
## 3. L'Enonce du Théorème -- Ce Qu'il Dit et Ne Dit Pas

### 3.1 L'enonce

**Théorème du libre arbitre (Conway-Kochen, 2006/2009).** Sous les axiomes SPIN, TWIN et MIN, la
reponse d'une particule a une mesure **ne peut pas etre une fonction** de l'information disponible
avant la mesure (l'état cache `lambda` et les choix passes).

Dit autrement, dans le langage des auteurs : *si les experimentateurs sont libres (leurs choix ne
sont pas fonction du passe), alors les particules le sont aussi.*

### 3.2 Ce que le théorème NE dit PAS (avertissement pedagogique)

C'est ici que la plupart des malentendus surgissent. Le mot "libre arbitre" est une **définition
mathematique**, pas une these philosophique ou neuroscientifique.

| Le théorème DIT | Le théorème NE dit PAS |
|-----------------|------------------------|
| La reponse d'une particule n'est pas une fonction de l'état cache anterieur | Que les humains possedent un libre arbitre metaphysique |
| Sous SPIN+TWIN+MIN, le determinisme a variables cachees est exclu | Que l'indeterminisme equivaut a la liberte morale |
| C'est un théorème **conditionnel** : "si les experimentateurs sont libres, alors..." | Que les experimentateurs SONT libres (c'est une hypothese) |
| "Libre" = "non determine par le passe" (def. technique) | Que la conscience joue un rôle |

Conway insistait : le théorème **transfere** une hypothese de liberte des experimentateurs vers les
particules. Il ne **créé** pas de liberte ex nihilo. Sa portee est l'**exclusion d'une classe de
théories déterministes**, exactement comme Kochen-Specker exclut les variables cachees non-contextuelles.

### 3.3 L'argument en deux temps

1. **Une particule.** Une reponse déterministe respectant SPIN serait, pour chaque `lambda`, une
   coloration valide des 18 vecteurs. Kochen-Specker dit qu'il n'en existe aucune. Contradiction.
2. **Deux particules.** Par TWIN, Alice et Bob partagent la même fonction de reponse. Par SPIN
   (cote Alice), cette fonction retombe sur le cas a une particule. Même contradiction.


In [3]:
# Illustration de la structure logique de l'argument en deux temps.
# (Le contenu formel est dans FreeWillTheorem.lean ; ici, on trace la logique.)

def two_particle_model_is_consistent(alice_resp, bob_resp):
    """TWIN + SPIN pour un modele deterministe a deux particules (un état cache)."""
    twin_ok = (alice_resp == bob_resp)        # TWIN : meme reponse sur memes directions
    spin_alice = is_valid_coloring(alice_resp)  # SPIN cote Alice
    spin_bob = is_valid_coloring(bob_resp)      # SPIN cote Bob
    return twin_ok and spin_alice and spin_bob

print("Argument en deux temps (trace logique) :")
print("  [TWIN]  Alice et Bob partagent la meme fonction de reponse")
print("  [SPIN]  cette fonction doit etre une coloration valide des 18 vecteurs")
print("  [KS]    aucune coloration valide n'existe  =>  CONTRADICTION")
print()
print("Formellement, free_will_theorem se reduit a fwt_single_particle,")
print("qui se reduit a kochen_specker. C'est ce que verifie la section 5.")


Argument en deux temps (trace logique) :
  [TWIN]  Alice et Bob partagent la meme fonction de reponse
  [SPIN]  cette fonction doit etre une coloration valide des 18 vecteurs
  [KS]    aucune coloration valide n'existe  =>  CONTRADICTION

Formellement, free_will_theorem se reduit a fwt_single_particle,
qui se reduit a kochen_specker. C'est ce que verifie la section 5.


### Interpretation : la mecanique de la contradiction

La trace illustre le raisonnement par l'absurde : si les reponses sont fonction de la direction (determinisme), alors TWIN impose une correlation qui viole SPIN. Le théorème ne dit pas que le libre arbitre *existe* -- il dit que le determinisme est **incompatible** avec les axiomes quantiques. C'est un résultat de non-existence, pas d'existence.

| Étape | Axiome | Consequence |
|-------|--------|-------------|
| TWIN | Particules intriquees, mêmes reponses | Fonction de reponse partagee |
| SPIN | Un seul "1" par base orthogonale | Doit etre une coloration KS valide |
| KS | Aucune coloration valide n'existe | Contradiction |

> **Note technique** : le théorème est **conditionnel**. Si l'un des trois axiomes est relaxe, la contradiction disparait.

### Exemple guide : formaliser la contrainte MIN (rendu TP PR #25531)

**Crédit** : rendu de TP intégré ici en exemple guidé — PR #25531 (Caroline et Juliana, TP 2). L'énoncé d'origine reste conservé ci-dessous.

**Enonce** : écrire une fonction Python qui vérifie que la reponse d'Alice ne depend pas de la direction choisie par Bob, en iterant sur les paires `(direction_alice, direction_bob)`. Montrer que toute violation de MIN implique une correlation interdite par SPIN.

**Indices** :
1. Simuler les reponses pour différentes paires de directions ; vérifier que la reponse d'Alice pour la direction `d_a` est la même quel que soit le choix `d_b` de Bob.
2. Définir une fonction `check_min(alice_func, bob_func, directions)` qui itere sur les paires de directions.
3. Pour chaque `(d_a, d_b)`, vérifier que `alice_func(d_a)` ne varie pas quand `d_b` change.
4. Si MIN est viole, montrer que cela produit une correlation contredisant SPIN.

**Ce que montre l'exemple** : deux modèles simulés — un modèle où chacun ne regarde que sa propre direction (MIN respectée, test passé) et un modèle où la réponse d'Alice « sent » le choix de Bob (MIN violée, contre-exemple exhibé avec les valeurs exactes).


In [4]:
# Exemple guide : verifier la contrainte MIN sur des modeles simules (rendu TP PR #25531).
# Exercice 2 :
directions = range(18)  # les 18 directions de mesure

def check_min(alice_func, bob_func, directions):
    """Renvoie (True, None) si MIN est respectée, sinon (False, contre-exemple)."""
    for d_a in directions:
        reference = alice_func(d_a, 0)               # réponse d'Alice quand Bob choisit 0
        for d_b in directions:
            if alice_func(d_a, d_b) != reference:    # la réponse change selon Bob
                return False, ("Alice", d_a, d_b)
    for d_b in directions:
        reference = bob_func(d_b, 0)                 # même test côté Bob
        for d_a in directions:
            if bob_func(d_b, d_a) != reference:
                return False, ("Bob", d_b, d_a)
    return True, None

# Modèle qui respecte MIN : chacun répond selon sa direction seulement
alice_ok = lambda d_a, d_b: d_a % 2
bob_ok   = lambda d_b, d_a: d_b % 2
# Modèle qui viole MIN : Alice "sent" le choix de Bob
alice_bad = lambda d_a, d_b: (d_a + d_b) % 2

min_satisfied, _ = check_min(alice_ok, bob_ok, directions)
print("Modèle ok  :", min_satisfied)                       # True

ok, (qui, d, d_autre) = check_min(alice_bad, bob_ok, directions)
print("Modèle bad :", ok)                                  # False
print(f"{qui}, direction {d} : répond {alice_bad(d, 0)} si l'autre choisit 0, "
      f"mais {alice_bad(d, d_autre)} s'il choisit {d_autre}")

# Interprétation :
# - Si la réponse d'Alice dépend du choix de Bob, sa valeur pour une direction n'est plus
#   un nombre fixe : on ne peut plus associer une valeur 0/1 à chaque vecteur,
#   donc il n'y a pas de coloration, et l'argument SPIN + KS ne s'applique plus.
# - Physiquement, cela voudrait dire que le choix de Bob agit instantanément sur Alice
#   (signal plus rapide que la lumière), ce que MIN interdit.


Modèle ok  : True
Modèle bad : False
Alice, direction 0 : répond 0 si l'autre choisit 0, mais 1 s'il choisit 1


### Lecture du résultat

Le modèle où chacun ne regarde que sa propre direction passe le test (`Modèle ok : True`). Le modèle où la réponse d'Alice dépend du choix de Bob échoue (`Modèle bad : False`) et le contre-exemple est explicite : direction 0, Alice répond 0 si Bob choisit 0 mais 1 s'il choisit 1. Sa « valeur » pour la direction 0 n'est plus un nombre fixe — plus de coloration 0/1 des vecteurs, donc l'argument SPIN + Kochen-Specker ne s'applique plus ; et physiquement, le choix de Bob agirait instantanément sur Alice, un signal plus rapide que la lumière.


### Exercice 2 (nouveau) : formaliser la contrainte TWIN

**Objectif.** Écrire une fonction Python qui vérifie la corrélation TWIN : lorsque Alice et Bob
mesurent la *même* direction, leurs réponses coïncident. En itérant sur les directions, montrer
qu'un modèle qui viole TWIN ne peut pas être représenté par une seule fonction partagée.

- *Indice* : pour chaque direction `d`, comparer `alice_func(d, d)` et `bob_func(d, d)`.
- *Étape 1* : définir `check_twin(alice_func, bob_func, directions)` qui itère sur les directions.
- *Étape 2* : construire un modèle qui respecte TWIN et un modèle qui le viole.
- *Étape 3* : exhiber le contre-exemple et interpréter — pourquoi TWIN est-elle essentielle pour que l'argument du théorème porte sur les *deux* particules ?


In [5]:
print("Exercice a completer")
twin_satisfied = None  # TODO etudiant : verifier la correlation TWIN sur les directions

# TODO etudiant : Etape 1 -- definir check_twin(alice_func, bob_func, directions)
# TODO etudiant : Etape 2 -- un modele qui respecte TWIN, un modele qui le viole
# TODO etudiant : Etape 3 -- exhiber le contre-exemple et interpreter


Exercice a completer


<a id="4"></a>
## 4. Le Port Lean 4 : Adossement au Code Réel

Toute la construction ci-dessus est formalisee dans `conway_lean/Conway/FreeWillTheorem.lean`, qui
s'appuie sur `conway_lean/Conway/KochenSpecker.lean` -- **le théorème est donc réellement
prouvé, pas seulement esquisse**.

La cellule suivante affiche les **définitions et théorèmes réels** extraits du `.lean` (pas une
paraphrase) -- c'est la source de verite. Puis on **exécute** le port : `grep -c sorry`,
`lake build`, et la trace de la chaîne de reduction.


In [6]:
# Affiche les declarations RÉELLES de FreeWillTheorem.lean (source de verite, pas une paraphrase).
fwt_path = WIN_LEAN_PROJECT / "Conway" / "FreeWillTheorem.lean"
src_lines = fwt_path.read_text(encoding="utf-8").splitlines()

decl_prefixes = ("abbrev ", "def ", "theorem ", "lemma ", "structure ", "inductive ")
print(f"FreeWillTheorem.lean : {len(src_lines)} lignes")
print("Declarations (inventaire) :")
for i, ln in enumerate(src_lines, 1):
    if ln.startswith(decl_prefixes):
        print(f"  L{i:>3}: {ln.strip()}")

def show_block(start_kw, max_lines=8):
    """Affiche le bloc d'une declaration (théorème + preuve) tel qu'ecrit dans le source."""
    for i, ln in enumerate(src_lines):
        if ln.strip().startswith(start_kw):
            print(f"\n--- {start_kw} (L{i+1}) ---")
            for j in range(i, min(i + max_lines, len(src_lines))):
                print(src_lines[j])
                if src_lines[j].strip() == "" and j > i + 1:
                    break
            return

show_block("theorem fwt_single_particle")
show_block("theorem free_will_theorem")


FreeWillTheorem.lean : 234 lignes
Declarations (inventaire) :
  L 75: abbrev HiddenState := ℕ
  L 83: abbrev DeterministicResponse := HiddenState → VecIdx → Bool
  L 93: def SatisfiesSPIN (f : DeterministicResponse) : Prop :=
  L107: theorem fwt_single_particle :
  L127: inductive Experimenter
  L148: abbrev TwoParticleResponse := HiddenState → Experimenter → VecIdx → Bool
  L156: def SatisfiesTWIN (f : TwoParticleResponse) : Prop :=
  L175: structure SatisfiesFWT (f : TwoParticleResponse) : Prop where
  L191: theorem free_will_theorem :

--- theorem fwt_single_particle (L107) ---
theorem fwt_single_particle :
    ¬ ∃ f : DeterministicResponse, SatisfiesSPIN f := by
  rintro ⟨f, hspin⟩
  exact kochen_specker ⟨f 0, hspin 0⟩


--- theorem free_will_theorem (L191) ---
theorem free_will_theorem :
    ¬ ∃ f : TwoParticleResponse, SatisfiesFWT f := by
  rintro ⟨f, hfwt⟩
  -- By SPIN for Alice: f(·, .alice) is a deterministic response
  -- satisfying SPIN. This contradicts the single-particle

### Vérification : de la declaration a la preuve complète

Les declarations ci-dessus définissent les types et lemmes du port Lean. Verifions maintenant que le port est **complète** : aucun `sorry` residuel, et le build compile.

In [7]:
# Critere d'acceptation : grep -c sorry visible dans un output.
import os

for fname in ["KochenSpecker.lean", "FreeWillTheorem.lean"]:
    fpath = os.path.join(str(WIN_LEAN_PROJECT), "Conway", fname)
    if os.path.exists(fpath):
        with open(fpath, "r", encoding="utf-8") as f:
            n = f.read().count("sorry")
        print(f"  {fname}: {n}")
    else:
        print(f"  {fname}: (fichier non trouve)")

print()
print("0 = aucune occurrence de 'sorry'. Les deux piliers sont formellement complets.")


  KochenSpecker.lean: 1
  FreeWillTheorem.lean: 0

0 = aucune occurrence de 'sorry'. Les deux piliers sont formellement complets.


In [8]:
# Exécution réelle du port : lake build des deux piliers.
build_timeout_s = 1200
plat = "natif" if is_native_platform() else "WSL"
print(f"lake build Conway.KochenSpecker Conway.FreeWillTheorem ({plat})...")
print("-" * 60)
rc, out, err = run_lake(LEAN_PROJECT, "build Conway.KochenSpecker Conway.FreeWillTheorem", timeout=build_timeout_s)
if out:
    print(out[-1800:] if len(out) > 1800 else out)
if err:
    print("STDERR:", err[-300:])
print()
print(f"Exit code : {rc}  (0 = SUCCESS)")
if rc == -1:
    print(f"TimeoutExpired apres {build_timeout_s}s (cache Mathlib non prechauffe).")


lake build Conway.KochenSpecker Conway.FreeWillTheorem (WSL)...
------------------------------------------------------------


Build completed successfully (3007 jobs).

Exit code : 0  (0 = SUCCESS)


### Interpretation : les deux piliers sont prouves

| Vérification | Résultat attendu | Signification |
|--------------|------------------|---------------|
| `grep -c sorry` KochenSpecker.lean | 0 | Le noyau combinatoire (parite) est complet |
| `grep -c sorry` FreeWillTheorem.lean | 0 | La reduction FWT -> KS est complète |
| `lake build` exit code | 0 | Les deux modules compilent réellement |

Le théorème de libre arbitre est donc **formellement prouvé** en Lean 4, et non simplement
esquisse. Precisions sur le compteur brut : l'unique occurrence du mot `sorry` dans
`KochenSpecker.lean` est un **commentaire** (ligne 107 : « Tous les `sorry` ont ete elimines »,
Epic #1453), pas une tactique -- le compteur de la cellule precedente compte la prose. Au
niveau des tactiques, les deux modules sont a zero `sorry`.

Deux sources attestent la compilation, et elles se lisent differemment. La **CI du lake**
(job `lean-conway` : `lake build` + `proof-integrity` sur `Conway.KochenSpecker` et
`Conway.FreeWillTheorem`, `fail-on-sorry: true`) verifie les deux modules a chaque PR qui
touche le lake. La cellule `lake build` ci-dessus, elle, depend de l'etat de l'environnement
local : un code de retour **-2** signifie que l'organe d'execution a refuse le lancement
(cache lake present sans epingle de backend) -- ce n'est ni un echec de compilation, ni une
preuve de reussite. La preuve tient en deux reductions -- c'est l'"elegance Conway" :
un enonce spectaculaire qui se ramene a un argument combinatoire de parite.

<a id="5"></a>
## 5. La Chaîne de Reduction Formelle

La force du port Lean est sa **concision** : le théorème spectaculaire `free_will_theorem` se
reduit au théorème a une particule, lui-même reduit a Kochen-Specker.
Verifions cette chaîne directement dans le source.


In [9]:
# Tracer la chaine de reduction : free_will_theorem -> fwt_single_particle -> kochen_specker.
src = (WIN_LEAN_PROJECT / "Conway" / "FreeWillTheorem.lean").read_text(encoding="utf-8")

chain = [
    ("free_will_theorem", "fwt_single_particle"),
    ("fwt_single_particle", "kochen_specker"),
]
print("Chaine de reduction (qui invoque qui dans les preuves) :\n")
for caller, callee in chain:
    idx = src.find(f"theorem {caller}")
    body = src[idx: idx + 600] if idx >= 0 else ""
    mark = "OK" if callee in body else "??"
    print(f"  [{mark}] {caller}  --reduit a-->  {callee}")
print()
print("free_will_theorem  ->  fwt_single_particle  ->  kochen_specker")
print("Profondeur de la chaine : 3 maillons. Le FWT est un corollaire (1 ligne) de KS.")


Chaine de reduction (qui invoque qui dans les preuves) :

  [OK] free_will_theorem  --reduit a-->  fwt_single_particle
  [OK] fwt_single_particle  --reduit a-->  kochen_specker

free_will_theorem  ->  fwt_single_particle  ->  kochen_specker
Profondeur de la chaine : 3 maillons. Le FWT est un corollaire (1 ligne) de KS.


### Interpretation : concision formelle vs complexite conceptuelle

La chaîne de reduction tient en deux tactiques `exact` Lean, alors que l'argument conceptuel couvre des pages. C'est l'elegance Conway : la formalisation capture l'essentiel. La reduction `free_will_theorem` s'appuie directement sur `fwt_single_particle` (le résultat a une particule) et `kochen_specker` (le noyau combinatoire).

| Maillon de la chaîne | Tactique Lean | Profondeur |
|----------------------|---------------|------------|
| `free_will_theorem` -> `fwt_single_particle` | `exact fwt_single_particle` | 1 |
| `fwt_single_particle` -> `kochen_specker` | `exact kochen_specker` | 2 |

Le théorème spectaculaire est donc un **corollaire direct** de Kochen-Specker, formalise en deux reductions.

### Exemple guide : decrire la structure de la preuve Lean (rendu TP PR #25531)

**Crédit** : rendu de TP intégré ici en exemple guidé — PR #25531 (Caroline et Juliana, TP 2). L'énoncé d'origine reste conservé ci-dessous.

**Enonce** : en vous basant sur le source affiche dans la section 4, decrire en francais la structure de la preuve `free_will_theorem`. Identifier chaque tactique utilisee et son rôle. Completer le tableau ci-dessous.

**Indices** :
1. Regarder les tactiques `exact`, `apply`, `intro`, `cases`, `have` dans le source Lean affiche en section 4.
2. Relever chaque tactique dans le bloc `theorem free_will_theorem`.
3. Pour chaque tactique, décrire son rôle dans le raisonnement.
4. Compléter le dictionnaire `proof_analysis` avec la correspondance tactique -> rôle.

**Ce que montre l'exemple** : la preuve `free_will_theorem` tient en trois tactiques — une contradiction construite pas à pas, chaque ligne du tableau pointant la ligne exacte du source (`FreeWillTheorem.lean`).


In [10]:
# Exemple guide : tableau d'analyse de la preuve free_will_theorem (rendu TP PR #25531).
proof_analysis = {
    "rintro ⟨f, hfwt⟩": (
        "Preuve par l'absurde : on suppose qu'il existe une f et on récupère "
        "hfwt, la preuve que f satisfait les axiomes (SPIN et TWIN).",
        193,
    ),
    "have hspin_alice := hfwt.spin .alice": (
        "On extrait des axiomes uniquement SPIN appliqué à Alice : "
        "f(·, alice) est une réponse déterministe qui satisfait SPIN.",
        196,
    ),
    "exact fwt_single_particle ⟨(f · .alice), hspin_alice⟩": (
        "On applique le théorème à une particule (¬∃ f, SatisfiesSPIN f) à ce "
        "témoin : contradiction, donc le but est démontré.",
        197,
    ),
}

print(f"{'Tactique':<55} | Ligne | Rôle")
for tactique, (role, ligne) in proof_analysis.items():
    print(f"{tactique:<55} | L{ligne:<4} | {role}")


Tactique                                                | Ligne | Rôle
rintro ⟨f, hfwt⟩                                        | L193  | Preuve par l'absurde : on suppose qu'il existe une f et on récupère hfwt, la preuve que f satisfait les axiomes (SPIN et TWIN).
have hspin_alice := hfwt.spin .alice                    | L196  | On extrait des axiomes uniquement SPIN appliqué à Alice : f(·, alice) est une réponse déterministe qui satisfait SPIN.
exact fwt_single_particle ⟨(f · .alice), hspin_alice⟩   | L197  | On applique le théorème à une particule (¬∃ f, SatisfiesSPIN f) à ce témoin : contradiction, donc le but est démontré.


### Lecture du résultat

Le tableau relève les trois tactiques de `theorem free_will_theorem` (aux lignes L193-L197 de `FreeWillTheorem.lean`) : `rintro` ouvre la contradiction en détruisant l'existencielle, `have` isole l'instance SPIN côté Alice, et `exact` conclut en invoquant `fwt_single_particle` — la preuve à une particule — sur le témoin construit. Trois lignes suffisent : tout le travail est déjà dans les théorèmes en aval, c'est la chaîne de réduction tracée en section 5.


### Exercice 3 (nouveau) : décrire la preuve `fwt_single_particle`

**Objectif.** En vous basant sur le source affiché dans la section 4, décrire en français la
structure de la preuve `fwt_single_particle` — l'étape 1 du théorème, la réduction à
Kochen-Specker. Identifier chaque tactique utilisée et son rôle, puis compléter le tableau.

- *Indice* : la preuve tient en deux tactiques ; regarder ce que fait le témoin `⟨f 0, hspin 0⟩` (pourquoi instancier à `0` suffit).
- *Étape 1* : relever chaque tactique dans le bloc `theorem fwt_single_particle`.
- *Étape 2* : pour chaque tactique, décrire son rôle dans le raisonnement.
- *Étape 3* : compléter le dictionnaire `proof_analysis_single` avec la correspondance tactique -> rôle.


In [11]:
print("Exercice a completer")
# TODO etudiant : completer le tableau d'analyse de la preuve fwt_single_particle
# | Tactique | Role dans la preuve | Ligne approx. |
# |----------|---------------------|---------------|
# | ...      | ...                 | ...           |
proof_analysis_single = None  # TODO etudiant : dictionnaire tactique -> (role, ligne)


Exercice a completer


<a id="6"></a>
## 6. La Non-Localite Quantitative : la Frontiere CHSH

Le théorème du libre arbitre s'appuie sur la **non-localite quantique** : la correlation des
particules jumelles (TWIN) ne s'explique par aucune fonction de reponse locale. Sa forme
quantitative canonique est l'inégalité **CHSH** (Clauser, Horne, Shimony, Holt, 1969) : dans tout
modele local classique, le score de correlation est borne par 2 en valeur absolue -- alors que la
mecanique quantique atteint 2*sqrt(2)* (borne de Tsirelson).

Les trois tranches CHSH sont compilees par le lac ; cette section rejoue l'enumeration de la
frontiere en Python, puis inventorie les declarations réelles du source Lean.

Le lac `conway_lean` prouve les deux moities classiques de cette frontiere, puis la franchit :

- `Conway/CHSH.lean` (epic #13106, tranche deterministe) : toute strategie locale **deterministe**
  a un score de valeur absolue **exactement 2** -- `Conway.CHSH.classical_abs_score`, démontre par
  enumeration noyau (decide) des 16 assignations, avec sa forme usuelle `classical_bound` ;
- `Conway/CHSHRandomized.lean` (tranche randomisee) : la frontiere tient pour les strategies
  **randomisees** -- toute combinaison convexe de profils garde un score espere borne par 2
  (`Conway.CHSHRandomized.randomized_bound`) ;
- `Conway/CHSHQuantum.lean` (tranche quantique) : la borne de Tsirelson $2\sqrt{2}$ est
  formalisee -- `Conway.CHSHQuantum.tsirelson_bound` importe le théorème de Mathlib
  (`Mathlib.Algebra.Star.CHSH.tsirelson_inequality`, preuve noyau par somme de carrés) sous les
  hypothèses exactes d'algèbre étoilée ordonnée, et le module prouve localement la reecriture
  $\sqrt{2}^3 = 2\sqrt{2}$ (`sqrt_two_cubed`) et l'ecart $2 < 2\sqrt{2}$
  (`classical_quantum_gap`).

Ce qui reste ouvert est desormais plus fin que « la formaliser demande des observables
hermitiennes et une norme d'operateur » : la borne importee est un **majorant unilateral**, et
le module ne construit pas l'etat quantique qui la sature -- pas de matrices de Pauli, pas
d'interpretation probabiliste complete, pas de borne bilaterale en norme d'operateur. Ces
limites sont declarees dans l'en-tete du module, et le registre de la section suivante en garde
la trace.

In [12]:
# Miroir numérique de l'enumeration Lean : les 16 profils deterministes.
from itertools import product

def outcome_value(o):
    # Miroir python de Conway.CHSH.Outcome.value : negative = -1, positive = +1.
    return -1 if o == "N" else +1

def score_chsh(a0, a1, b0, b1):
    # Miroir python de Conway.CHSH.score : a0*b0 + a0*b1 + a1*b0 - a1*b1.
    return a0 * b0 + a0 * b1 + a1 * b0 - a1 * b1

rows = []
for a0, a1, b0, b1 in product(("N", "P"), repeat=4):
    s = score_chsh(outcome_value(a0), outcome_value(a1),
                   outcome_value(b0), outcome_value(b1))
    rows.append((f"{a0}{a1}{b0}{b1}", s))

print("16 profils deterministes (a0 a1 b0 b1) :")
for prof, s in rows:
    print(f"  {prof}  score = {s:+d}")
abs_scores = sorted({abs(s) for _, s in rows})
print(f"\nValeurs de |score| observees : {abs_scores}")
print(f"Max |score| = {max(abs_scores)}  (frontiere classique deterministe)")

16 profils deterministes (a0 a1 b0 b1) :
  NNNN  score = +2
  NNNP  score = +2
  NNPN  score = -2
  NNPP  score = -2
  NPNN  score = +2
  NPNP  score = -2
  NPPN  score = +2
  NPPP  score = -2
  PNNN  score = -2
  PNNP  score = +2
  PNPN  score = -2
  PNPP  score = +2
  PPNN  score = -2
  PPNP  score = -2
  PPPN  score = +2
  PPPP  score = +2

Valeurs de |score| observees : [2]
Max |score| = 2  (frontiere classique deterministe)


### Interpretation : |score| = 2 partout, et Lean le prouve

L'enumeration python confirme ce que `Conway.CHSH.classical_abs_score` démontre dans le noyau
Lean : **les 16 strategies deterministes ont toutes un score de valeur absolue exactement 2** --
la frontiere n'est pas une borne atteinte par certaines strategies, c'est la valeur **commune a
toutes**. Le theorem `classical_bound` en est la forme usuelle (|score| <= 2), et
`score_factorization` explique **pourquoi** : le score se reecrit
`a0*(b0+b1) + a1*(b0-b1)`, et pour des reponses binaires (+-1), `b0+b1` et `b0-b1` ne peuvent pas
etre non nuls tous les deux -- un seul des deux termes contribue, de valeur absolue 2.

In [13]:
# Inventaire des declarations RÉELLES des trois modules CHSH (source de verite, pas une paraphrase).
decl_prefixes = ("abbrev ", "def ", "theorem ", "lemma ", "structure ", "inductive ")

for fname in ["CHSH.lean", "CHSHRandomized.lean", "CHSHQuantum.lean"]:
    fpath = WIN_LEAN_PROJECT / "Conway" / fname
    src_lines = fpath.read_text(encoding="utf-8").splitlines()
    n_sorry = sum(1 for l in src_lines if "sorry" in l)
    print(f"{fname} : {len(src_lines)} lignes, {n_sorry} occurrence(s) de 'sorry'")
    for i, ln in enumerate(src_lines, 1):
        if ln.startswith(decl_prefixes):
            print(f"  L{i:>3}: {ln.strip()[:100]}")
    print()
print("0 occurrence en TACTIQUE = la frontiere est complete (deterministe, randomisee ET quantique).")
print("Une occurrence isolee dans l'en-tete du module compte la PROSE, pas une tactique :")
print("c'est le piege du compteur brut, tranche par count_code_sorry.py (mode real).")

CHSH.lean : 88 lignes, 0 occurrence(s) de 'sorry'
  L 33: inductive Outcome
  L 39: def Outcome.value : Outcome → ℤ
  L 44: theorem Outcome.value_sq (outcome : Outcome) : outcome.value ^ 2 = 1 := by
  L 52: def score (a₀ a₁ b₀ b₁ : Outcome) : ℤ :=
  L 58: theorem score_factorization (a₀ a₁ b₀ b₁ : Outcome) :
  L 70: theorem classical_abs_score (a₀ a₁ b₀ b₁ : Outcome) :
  L 75: theorem classical_bound (a₀ a₁ b₀ b₁ : Outcome) :

CHSHRandomized.lean : 176 lignes, 0 occurrence(s) de 'sorry'
  L 43: abbrev Profile := CHSH.Outcome × CHSH.Outcome × CHSH.Outcome × CHSH.Outcome
  L 53: def Profile.score (p : Profile) : ℤ :=
  L 58: theorem Profile.abs_score (p : Profile) : |Profile.score p| = 2 := by
  L 64: abbrev Strategy := Profile → ℚ
  L 68: def expectedScore (μ : Strategy) : ℚ :=
  L 73: theorem Profile.abs_score_rat (p : Profile) : |(Profile.score p : ℚ)| = 2 := by
  L 82: theorem randomized_bound (μ : Strategy)
  L107: def pPos : Profile := (.positive, .positive, .positive, .positive)
 

### Interpretation : la randomite partagee ne brise pas la frontiere

`Conway.CHSHRandomized.randomized_bound` etend le resultat aux strategies **randomisees** : une
strategie est une famille de poids rationnels sur les 16 profils (le type `Strategy`), son score
espere `expectedScore` est la combinaison convexe des scores deterministes. La preuve combine
l'inegalite triangulaire (valeur absolue d'une somme <= somme des valeurs absolues) avec le fait
que chaque profil porte \|score\| = 2 (`Profile.abs_score`) : la convexite des poids
(`h_nonneg`, `h_total`) ramene le tout a une somme de poids fois 2, donc \|E[score]\| <= 2.

C'est exactement la structure qui rend le theoreme du libre arbitre **conditionnel** : aucune
theorie locale -- meme randomisee, meme avec randomite partagee -- ne reproduit la correlation
quantique 2*sqrt(2)*. La frontiere ne peut etre franchie qu'en sortant du modele local (reponses
quantiques), ce que FWT capture par ses axiomes SPIN + TWIN + MIN.

### Exemple guide : une strategie randomisee sur la frontiere (rendu TP PR #25531)

**Crédit** : rendu de TP intégré ici en exemple guidé — PR #25531 (Caroline et Juliana, TP 2). L'énoncé d'origine reste conservé ci-dessous.

**Enonce** : construisez en python une strategie randomisee (poids sur les 16 profils, somme = 1) dont le
score espere atteint la frontière |E[score]| = 2, puis vérifiez sur quelques exemples que melanger
deux profils de scores opposes rapproche E[score] de 0. Indice : tout profil deterministe convient
pour la frontière -- la subtilité est le miroir exact d'`expectedScore`.

**Ce que montre l'exemple** : le miroir exact d'`expectedScore` en arithmétique exacte (`Fraction`) — un profil seul atteint la frontière E = 2, et le mélange du profil +2 avec le profil -2 ramène linéairement l'espérance vers 0 (poids 3/4 → E = 1, poids 1/2 → E = 0), sans jamais sortir de la frontière.


In [14]:
# Exemple guide : miroir exact d'expectedScore en arithmetique exacte (rendu TP PR #25531).
from itertools import product
from fractions import Fraction

def outcome_value(o):
    return -1 if o == "N" else +1

def score_chsh(a0, a1, b0, b1):
    return a0*b0 + a0*b1 + a1*b0 - a1*b1

profiles = ["".join(p) for p in product("NP", repeat=4)]
score = {p: score_chsh(*(outcome_value(c) for c in p)) for p in profiles}

def expected_score(strategy):
    """Score espéré d'une stratégie randomisée : moyenne pondérée des profils."""
    assert sum(strategy.values()) == 1, "les poids doivent sommer à 1"
    assert all(w >= 0 for w in strategy.values())
    return sum(w * score[p] for p, w in strategy.items())

# Partie A : un seul profil (poids 1) -> sur la frontière |E| = 2
print("A :", expected_score({"PPPP": Fraction(1)}))

# Partie B : mélanger deux profils de scores opposés
pos = next(p for p in profiles if score[p] == +2)
neg = next(p for p in profiles if score[p] == -2)
print("profil +2 :", pos, "| profil -2 :", neg)
for w in [Fraction(1), Fraction(3, 4), Fraction(1, 2)]:
    E = expected_score({pos: w, neg: 1 - w})
    print(f"poids {w} sur {pos}, {1-w} sur {neg} -> E = {E}, |E| <= 2 : {abs(E) <= 2}")


A : 2
profil +2 : NNNN | profil -2 : NNPN
poids 1 sur NNNN, 0 sur NNPN -> E = 2, |E| <= 2 : True
poids 3/4 sur NNNN, 1/4 sur NNPN -> E = 1, |E| <= 2 : True
poids 1/2 sur NNNN, 1/2 sur NNPN -> E = 0, |E| <= 2 : True


### Lecture du résultat

La sortie montre `A : 2` : un profil déterministe seul (PPPP, poids 1) atteint déjà la frontière |E| = 2. Ensuite, le mélange du profil +2 (NNNN) avec le profil -2 (NNPN) fait descendre l'espérance exactement linéairement : poids 3/4 → E = 1, poids 1/2 → E = 0 — l'arithmétique `Fraction` donne des valeurs exactes, sans erreur de flottant. À chaque étape |E| <= 2 : la randomisation déplace l'espérance *à l'intérieur* de la frontière, elle ne la franchit jamais — exactement ce que prouve `Conway.CHSHRandomized.randomized_bound`.


### Exercice 4 (nouveau) : le miroir expérimental de `randomized_bound`

**Objectif.** Vérifier expérimentalement que *toute* stratégie randomisée reste dans la frontière
|E[score]| <= 2, comme le prouve formellement `Conway.CHSHRandomized.randomized_bound`.
Échantillonnez des stratégies aléatoires (poids positifs de somme 1 sur les 16 profils) et
vérifiez qu'aucune ne dépasse 2 en valeur absolue.

- *Indice* : tirer des poids positifs uniformément, puis les normaliser par leur somme.
- *Étape 1* : construire une centaine de stratégies randomisées valides par échantillonnage.
- *Étape 2* : calculer leur score espéré avec le miroir d'`expected_score` ci-dessus.
- *Étape 3* : constater |E| <= 2 dans tous les cas et relier l'observation à la convexité (combinaison convexe de valeurs toutes en ±2).


In [15]:
print("Exercice a completer")
max_abs_E = None  # TODO etudiant : max de |E[score]| observe sur les strategies echantillonnees

# TODO etudiant : Etape 1 -- echantillonner des strategies (poids > 0, somme = 1) sur les 16 profils
# TODO etudiant : Etape 2 -- calculer E[score] de chacune avec le miroir d'expected_score
# TODO etudiant : Etape 3 -- verifier |E| <= 2 partout et interpreter (convexite des ±2)


Exercice a completer


<a id="7"></a>
## 7. Extensibilite : Vers d'Autres Résultats de Conway

Le workspace `conway_lean` est concu comme un **hommage extensible** a l'oeuvre de Conway. Le
théorème de libre arbitre y cotoie d'autres formalisations (Game of Life, Doomsday, Look-and-Say,
FRACTRAN, Nim, ange et diable). Cette section pose un **registre** des résultats et de leur statut
de port, que l'on peut faire grandir.

Pistes d'extension naturelles a partir du FWT :
- **Strong Free Will Theorem (2009)** : la version avec MIN (déjà amorcee dans le docstring
  `Corollary: the "strong" form` de `FreeWillTheorem.lean`).
- **Coordonnees R^4 explicites** : remplacer la structure combinatoire abstraite de
  `contextMembers` par les 18 vecteurs réels + preuve d'orthogonalite (actuellement implicite).
- **Autres théorèmes de contextualite** : Peres-33, carré magique de Mermin-Peres.


In [16]:
# Registre extensible des résultats Conway et de leur statut de port Lean.
# (Verifie dynamiquement la presence des modules + leur compte de sorry.)
conway_results = [
    # (nom affiche, module .lean ou None, statut conceptuel)
    ("Kochen-Specker (Cabello 18)", "Conway/KochenSpecker.lean",   "PROUVE"),
    ("Free Will Theorem",           "Conway/FreeWillTheorem.lean", "PROUVE"),
    ("CHSH deterministe (|score| = 2)", "Conway/CHSH.lean",            "PROUVE"),
    ("CHSH randomise (|E| <= 2)",       "Conway/CHSHRandomized.lean",   "PROUVE"),
    ("Tsirelson quantique (2*sqrt(2))", "Conway/CHSHQuantum.lean",      "PROUVE"),
    ("Game of Life (B3/S23)",       "Conway/Life.lean",            "PROUVE"),
    ("Strong FWT (MIN, 2009)",      None,                          "PISTE"),
    ("Coordonnees R^4 explicites",  None,                          "PISTE"),
]

print(f"{'Resultat Conway':<32} | {'Module':<28} | {'Statut':<8} | sorry")
print("-" * 86)
for name, module, statut in conway_results:
    if module:
        p = WIN_LEAN_PROJECT / module
        if p.exists():
            n_sorry = sum(1 for l in p.read_text(encoding="utf-8").splitlines() if "sorry" in l)
            mod_disp, sorry_disp = module.replace("Conway/", ""), str(n_sorry)
        else:
            mod_disp, sorry_disp = module.replace("Conway/", "") + " (absent)", "-"
    else:
        mod_disp, sorry_disp = "(a creer)", "-"
    print(f"{name:<32} | {mod_disp:<28} | {statut:<8} | {sorry_disp}")
print()
print("Pour etendre : ajouter une entree ci-dessus + le module .lean dans conway_lean/Conway/.")

Resultat Conway                  | Module                       | Statut   | sorry
--------------------------------------------------------------------------------------
Kochen-Specker (Cabello 18)      | KochenSpecker.lean           | PROUVE   | 1
Free Will Theorem                | FreeWillTheorem.lean         | PROUVE   | 0
CHSH deterministe (|score| = 2)  | CHSH.lean                    | PROUVE   | 0
CHSH randomise (|E| <= 2)        | CHSHRandomized.lean          | PROUVE   | 0
Tsirelson quantique (2*sqrt(2))  | CHSHQuantum.lean             | PROUVE   | 1
Game of Life (B3/S23)            | Life.lean                    | PROUVE   | 0
Strong FWT (MIN, 2009)           | (a creer)                    | PISTE    | -
Coordonnees R^4 explicites       | (a creer)                    | PISTE    | -

Pour etendre : ajouter une entree ci-dessus + le module .lean dans conway_lean/Conway/.


### Interpretation : état du port Conway

Sur les 8 résultats Conway du registre, 6 sont **PROUVES** en Lean (le noyau Kochen-Specker, la
reduction FWT, les trois tranches CHSH et le Game of Life) et 2 sont en **PISTE** (Strong FWT,
coordonnees R^4). Le théorème du libre arbitre (Conway-Kochen 2006/2009) est le résultat central
du port -- sa preuve complète est un jalon formel pour la communaute.

| Résultat | Statut | Commentaire |
|----------|--------|-------------|
| Kochen-Specker (Cabello 18) | PROUVÉ | Noyau combinatoire |
| Free Will Theorem | PROUVÉ | Reduction FWT -> KS |
| CHSH déterministe (\|score\| = 2) | PROUVÉ | Enumeration noyau des 16 profils |
| CHSH randomisé (\|E\| ≤ 2) | PROUVÉ | Convexite des strategies locales |
| Tsirelson quantique (2√2) | PROUVÉ | Import Mathlib + ecarts locaux ; saturation ouverte |
| Game of Life (B3/S23) | PROUVÉ | Règles d'evolution |
| Strong FWT (MIN, 2009) | PISTE | Module a créer |
| Coordonnees R^4 explicites | PISTE | Vecteurs réels + orthogonalite |

> **Note technique** : le workspace `conway_lean` est extensible par design. Ajouter un résultat = créer le module `.lean` + ajouter une entrée au registre.

<a id="8"></a>
## 8. Exercices

Des exercices pour approfondir. Les cellules de code sont des **stubs** a completer : le notebook
s'exécute de bout en bout même sans les resoudre. Comparez vos solutions aux exemples guides des
sections précédentes.


### Exemple guide -- SPIN comme coloration

_Resolu par le groupe oceane.xiang & mehdi.robardet (contribution PR #2287)._

**Objectif.** Ecrire une fonction `response_is_spin(response)` qui vérifie qu'une reponse
déterministe donnée (liste de 18 valeurs 0/1) respecte l'axiome SPIN, c'est-a-dire qu'elle est une
coloration valide : exactement un `1` par contexte.

- *Indice* : reutiliser la liste `contexts` définie en section 2.1.
- *Étape 1* : pour chaque contexte, compter le nombre de `1`.
- *Étape 2* : renvoyer `True` si et seulement si chaque contexte contient exactement un `1`.


In [17]:
# Exemple guide 1 : SPIN comme coloration valide.
def response_is_spin(response):
    """Renvoie True si `response` (liste de 18 entiers 0/1) respecte SPIN.

    Demarche :
      Etape 1 : pour chaque contexte de `contexts`, compter le nombre de 1.
      Etape 2 : renvoyer True ssi chaque contexte contient exactement un 1.
    """
    for ctx in contexts:
        if sum(response[i] for i in ctx) != 1:
            return False
    return True


### Exemple guide -- La contrainte TWIN

_Resolu par le groupe oceane.xiang & mehdi.robardet (contribution PR #2287)._

**Objectif.** Completer `twin_holds(alice_resp, bob_resp)` qui modelise l'effet de l'axiome TWIN :
vérifier que les reponses d'Alice et Bob coincident (mêmes valeurs sur toutes les directions),
prerequis pour ramener le cas a deux particules au cas a une particule.

- *Indice* : TWIN impose `f(state, alice, dir) == f(state, bob, dir)` pour toute direction.
- *Étape 1* : comparer les deux listes terme a terme (18 directions).
- *Étape 2* : renvoyer `True` si elles sont identiques sur toutes les directions.


In [18]:
# Exemple guide 2 : la contrainte TWIN.
def twin_holds(alice_resp, bob_resp):
    """Renvoie True si TWIN est respecte : Alice et Bob ont la meme reponse partout.

    Demarche :
      Etape 1 : comparer alice_resp et bob_resp terme a terme (18 directions).
      Etape 2 : renvoyer True ssi elles coincident sur toutes les directions.
    """
    return alice_resp == bob_resp


### Exemple guide -- Etendre le registre Conway

_Resolu par le groupe oceane.xiang & mehdi.robardet (contribution PR #2287)._

**Objectif.** Ajouter au registre `conway_results` (section 6) une nouvelle entrée decrivant un
résultat de Conway non encore porte, avec ses hypotheses. Puis decrire en une phrase le module Lean
qu'il faudrait créer et sa stratégie de preuve.

- *Indice* : par exemple le "Strong Free Will Theorem" (2009) ou le carré magique de Mermin-Peres.
- *Étape 1* : définir `new_result = (nom, module_cible_ou_None, statut)`.
- *Étape 2* : ecrire dans `extension_note` une phrase decrivant axiomes et stratégie de preuve.


In [19]:
# Exemple guide 3 : etendre le registre Conway.
# Etape 1 : une nouvelle entrée de registre (nom, module cible ou None, statut).
new_result = ("Mermin-Peres magic square", None, "PISTE")

# Etape 2 : axiomes / strategie de preuve en une phrase.
extension_note = (
    "Formalisation du carré magique de Mermin-Peres demontrant la contextualite "
    "quantique via une grille de 9 observables sans assignation de valeurs "
    "predeterminees consistante."
)


***

### Exemple guide : aucune coloration déterministe ne satisfait SPIN (rendu TP PR #25531)

**Crédit** : rendu de TP intégré ici en exemple guidé — PR #25531 (Caroline et Juliana, TP 2). L'énoncé d'origine reste conservé ci-dessous.

**Enonce** : en vous appuyant sur l'exemple guide `response_is_spin` ci-dessus, montrez (par recherche
exhaustive ou par un argument de parite) qu'il n'existe **aucune** reponse déterministe de
18 bits validant SPIN sur l'ensemble des contextes. C'est exactement le contenu du théorème
de Kochen-Specker (1967) : l'impossibilite d'une assignation de valeurs predeterminees
consistante.

**Indices** :
1. Il y a `2**18 = 262144` reponses possibles ; une recherche par force brute est faisable.
2. Enumerer les reponses candidates avec `itertools.product([0, 1], repeat=18)`.
3. Compter combien satisfont `response_is_spin` ; conclure que ce nombre vaut 0.

**Ce que montre l'exemple** : la solution des étudiantes double l'argument — un argument de parité instantané (chaque vecteur appartient à exactement 2 contextes, et il y a 9 contextes : un nombre impair de « 1 » comptés contexte par contexte ne peut pas être un compte doublement pair) puis l'énumération exhaustive des 262 144 réponses, qui confirme le zéro.


In [20]:
# Exemple guide : Kochen-Specker par argument de parite puis force brute (rendu TP PR #25531).
from itertools import product

def response_is_spin(response):
    for ctx in contexts:
        if sum(response[i] for i in ctx) != 1:
            return False
    return True

# Méthode 1 : argument de parité (instantané, aucune énumération)
# SPIN => exactement un "1" par contexte, donc 9 "1" comptés contexte par contexte (impair).
# Mais chaque vecteur est dans exactement 2 contextes : chaque "1" est compté 2 fois,
# donc le total doit être pair. Impair ≠ pair : contradiction.
chaque_vecteur_2_fois = all(sum(i in ctx for ctx in contexts) == 2 for i in range(18))
print("Chaque vecteur est dans 2 contextes :", chaque_vecteur_2_fois)   # True
print("Contextes :", len(contexts), "(impair) → total pair impossible")  # 9

# Méthode 2 : force brute sur les 2**18 = 262 144 réponses
# (response_is_spin s'arrête dès qu'un contexte est faux, ce qui garde le calcul rapide)
nb_valides = 0
for r in product([0, 1], repeat=18):
    if response_is_spin(list(r)):
        nb_valides += 1
print("Réponses respectant SPIN :", nb_valides)                         # 0 (≈ 0,2 s)

# Interprétation :
# - 0 réponse valide sur 262 144 : aucune assignation déterministe 0/1 n'existe.
# - C'est le théorème de Kochen-Specker. La parité l'explique : 9 est impair, 2 × (nombre de 1) est pair.
# - Avec TWIN (Alice et Bob ont la même réponse), le déterminisme est donc impossible.


Chaque vecteur est dans 2 contextes : True
Contextes : 9 (impair) → total pair impossible


Réponses respectant SPIN : 0


### Lecture du résultat

La sortie confirme les deux volets : chaque vecteur apparaît dans exactement 2 contextes (`True`) et les 9 contextes rendent le total des « 1 » à la fois impair (un par contexte) et pair (chaque « 1 » compté deux fois) — contradiction. L'énumération exhaustive des 262 144 réponses rend le même verdict : `Réponses respectant SPIN : 0`. C'est le théorème de Kochen-Specker : aucune assignation déterministe de valeurs ne satisfait SPIN, et avec TWIN le déterminisme des deux particules tombe avec celui d'une seule.


### Exercice (nouveau) : relâcher un seul contexte

En retirant **un seul** des 9 contextes de la liste `contexts`, l'argument de parité ne
s'applique plus. Enumerez les réponses déterministes qui satisfont SPIN sur les 8 contextes
restants : combien en existe-t-il ? Comparez avec le cas des 9 contextes (aucune) et
expliquez le rôle de l'imparité dans le théorème.

- *Indice* : `sub_contexts = contexts[:-1]` puis réutiliser la même énumération exhaustive.
- *Étape 1* : construire la liste des 8 contextes.
- *Étape 2* : compter les réponses valides pour cette sous-liste.
- *Étape 3* : interpréter — pourquoi la contradiction de parité disparaît-elle ?


In [21]:
print("Exercice a completer")
nb_valides_8 = None  # TODO etudiant : compter les colorations valides sur 8 contextes

# TODO etudiant : Etape 1 -- liste des 8 contextes (retirer le dernier de `contexts`)
# TODO etudiant : Etape 2 -- enumerer et compter les reponses valides sur ces 8 contextes
# TODO etudiant : Etape 3 -- interpreter par rapport a la parite (9 contextes = impair)


Exercice a completer


## Resume

### Concepts cles

| Concept | Définition | Rôle |
|---------|------------|------|
| **SPIN** | Carré du spin-1 = un seul "1" par base orthogonale | Pont vers Kochen-Specker (coloration valide) |
| **TWIN** | Particules intriquees : mêmes reponses sur axes paralleles | Force une fonction de reponse partagee |
| **MIN (FIN)** | Choix d'experimentateurs independants (separation espace) | Structurel dans la signature de type |
| **Libre arbitre (math.)** | La reponse n'est pas une fonction du passe | Conclusion conditionnelle du théorème |

### Le port Lean 4

| Module | Rôle |
|--------|------|
| `KochenSpecker.lean` | Noyau combinatoire (parite, 18 vecteurs Cabello) |
| `FreeWillTheorem.lean` | Reduction FWT -> KS (SPIN + TWIN + MIN structurel) |
| `CHSH.lean` | Frontiere CHSH deterministe (\|score\| = 2 exactement) |
| `CHSHRandomized.lean` | Frontiere CHSH randomisee (\|E[score]\| <= 2 par convexite) |
| `CHSHQuantum.lean` | Borne de Tsirelson 2√2 (import Mathlib + ecarts locaux ; saturation ouverte) |

Chaîne de reduction : `free_will_theorem` -> `fwt_single_particle` -> `kochen_specker` (3 maillons).

### Avertissement retenu

"Libre arbitre" est une **définition mathematique** (reponse non fonction du passe), un théorème
**conditionnel**, et l'exclusion d'une classe de théories déterministes -- **pas** une these sur le
libre arbitre humain ou la conscience.

### References

1. **J. H. Conway, S. Kochen**, "The Free Will Theorem", Found. Phys. 36 (2006), 1441-1473. arXiv:quant-ph/0604079.
2. **J. H. Conway, S. Kochen**, "The Strong Free Will Theorem", Notices AMS 56 (2009), 226-232. arXiv:0807.3286.
3. **A. Cabello, J. M. Estebaranz, G. Garcia-Alcaine**, "Bell-Kochen-Specker theorem: A proof with 18 vectors", Phys. Lett. A 212 (1996), 183-187.
4. **S. Kochen, E. P. Specker**, "The Problem of Hidden Variables in Quantum Mechanics", J. Math. Mech. 17 (1967), 59-87.

### Liens

- [Lean-13 Kochen-Specker](Lean-13-Kochen-Specker.ipynb) : le noyau combinatoire (table Cabello, argument de parite)
- [GOL-01 Conway Tribute](GOL/GOL-01-Regles-Vie-Lean.ipynb) : le Game of Life formel
- [Lean-16 Conway, l'homme et l'oeuvre](Lean-16-Conway-Homme-et-Oeuvre-Lean-Python.ipynb) : biographie et panorama

***

**Navigation** : [<< Lean-16b FRACTRAN](Lean-16b-Conway-FRACTRAN-Lean-Native.ipynb) | [Index](README.md) | [Lean-13f Correlation Bell du vide >>](Lean-13f-Correlation-Bell-Vide-Lean-Python.ipynb)